# Step 1: Get era5 surface and upper air data

In [4]:
import cdsapi # Copernicus/ECMWF's Climate Data Store API


# Grabbing 00Z data for the March 10th midwest Hail event
year = "2026"
month = "03"
day = "11"
time = "00:00"



# Grabbing all the variables I need on that day to get 500 hPa height and SBCAPE. Multi-pressure data so need to input pressure levels
dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "geopotential",
        "specific_humidity",
        "temperature"
    ],
    "year": [year],
    "month": [month],
    "day": [day],
    "time": [time],
    "pressure_level": [
        "100", "125", "150",
        "175", "200", "225",
        "250", "300", "350",
        "400", "450", "500",
        "550", "600", "650",
        "700", "750", "775",
        "800", "825", "850",
        "875", "900", "925",
        "950", "975", "1000"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

client = cdsapi.Client()
client.retrieve(dataset, request, "./Datasets/era5_3-11-2026_pressures.nc")

2026-03-29 19:10:11,382 INFO Request ID is 46dfc684-fe76-42e2-b17f-adb0b1188ae8
2026-03-29 19:10:11,525 INFO status has been updated to accepted
2026-03-29 19:11:01,959 INFO status has been updated to successful


83f7e95038c0ee1d4333cef8d28c9297.nc:   0%|          | 0.00/140M [00:00<?, ?B/s]

'./Datasets/era5_3-11-2026_pressures.nc'

In [6]:
# Grabbing single level data (PWAT, and surface profile for Surface Based CAPE calculation)
client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": ["reanalysis"],
        "variable": [
            "total_column_water_vapour",
            "surface_pressure",
            "2m_temperature",
            "2m_dewpoint_temperature",
        ],
        "year": [year],
        "month": [month],
        "day": [day],
        "time": [time],
        "data_format": "netcdf",
        "download_format": "unarchived",
    },
    "./Datasets/era5_3-11-2026_surface.nc"
)

2026-03-29 19:12:35,766 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-29 19:12:35,769 INFO Request ID is 2d43ceb2-1460-49eb-ab58-cc0b15843d89
2026-03-29 19:12:35,926 INFO status has been updated to accepted
2026-03-29 19:12:50,033 INFO status has been updated to running
2026-03-29 19:12:57,866 INFO status has been updated to successful


b333a9479702cf640e5753a2f22a68da.nc:   0%|          | 0.00/5.82M [00:00<?, ?B/s]

'./Datasets/era5_3-11-2026_surface.nc'

## Step 2: grab Precipitable Water (Total Column Water Vapor), and 500 hPa height

In [8]:
import xarray as xr

g0 = 9.80665

pl = xr.open_dataset("./Datasets/era5_3-11-2026_pressures.nc")
sl = xr.open_dataset("./Datasets/era5_3-11-2026_surface.nc")

z500 = pl["z"].sel(pressure_level=500) / g0

pwat = sl["tcwv"]

## Step 3: Calculate SBCAPE

In [18]:
import numpy as np
import xarray as xr
import metpy.calc as mpcalc
from metpy.units import units
from tqdm.notebook import tqdm  # use: from tqdm import tqdm  if not in Jupyter

plev_hpa = pl["pressure_level"].values.astype(float)


wlon = 220
elon = 305
nlat = 55
slat = 20

# ERA5 latitude is descending, so slice north -> south
pl_sub = pl.sel(
    longitude=slice(wlon, elon),
    latitude=slice(nlat, slat)
)

sl_sub = sl.sel(
    longitude=slice(wlon, elon),
    latitude=slice(nlat, slat)
)


# Pull out the lat/lon and pressure levels I need

plev_hpa = pl_sub["pressure_level"].values.astype(float)
lat = pl_sub["latitude"]
lon = pl_sub["longitude"]

# Remove the valid_time dimension, as we are only calculating for 1 day.
t_vals   = pl_sub["t"].squeeze("valid_time").values       
q_vals   = pl_sub["q"].squeeze("valid_time").values       
sp_vals  = sl_sub["sp"].squeeze("valid_time").values      
t2m_vals = sl_sub["t2m"].squeeze("valid_time").values     
d2m_vals = sl_sub["d2m"].squeeze("valid_time").values     

nlat = len(lat)
nlon = len(lon)

sbcape_arr = np.full((nlat, nlon), np.nan, dtype=np.float64)

total_points = nlat * nlon

with tqdm(total=total_points, desc="Calculating SBCAPE") as pbar:
    for j in range(nlat):
        for i in range(nlon):
            t_pl_k = t_vals[:, j, i]        # grab the full profile for each varible at the given lat lon
            q_pl = q_vals[:, j, i]
            sp_pa = sp_vals[j, i]
            t2m_k = t2m_vals[j, i]
            d2m_k = d2m_vals[j, i]

            # Skip bad grid points
            if (
                not np.isfinite(sp_pa)
                or not np.isfinite(t2m_k)
                or not np.isfinite(d2m_k)
            ):
                pbar.update(1)
                continue

            # Surface pressure from Pa to hPa
            sp_hpa = sp_pa / 100.0

            # Keep only levels above ground and with finite values
            valid = (
                np.isfinite(t_pl_k)
                & np.isfinite(q_pl)
                & np.isfinite(plev_hpa)
                & (plev_hpa < sp_hpa)
            )

            if valid.sum() < 2:
                pbar.update(1)
                continue

            try:
                # Calculate dewpoint on pressure levels
                td_pl = mpcalc.dewpoint_from_specific_humidity(
                    plev_hpa[valid] * units.hPa,
                    t_pl_k[valid] * units.kelvin,
                    q_pl[valid] * units.dimensionless,
                ).to("degC")

                # Build a surface-based profile
                p_prof = np.concatenate(([sp_hpa], plev_hpa[valid])) * units.hPa
                T_prof = np.concatenate(([t2m_k - 273.15], t_pl_k[valid] - 273.15)) * units.degC
                Td_prof = np.concatenate(([d2m_k - 273.15], td_pl.magnitude)) * units.degC

                # Sort everything from high pressure to low pressure
                order = np.argsort(p_prof.magnitude)[::-1]
                p_prof = p_prof[order]
                T_prof = T_prof[order]
                Td_prof = Td_prof[order]

                # remove any duplicates or unsorted values 
                keep = np.ones(len(p_prof), dtype=bool)
                keep[1:] = np.diff(p_prof.magnitude) < 0

                p_prof = p_prof[keep]
                T_prof = T_prof[keep]
                Td_prof = Td_prof[keep]

                if len(p_prof) < 3:
                    pbar.update(1)
                    continue
                # calculate SBCAPE
                sbcape, _ = mpcalc.surface_based_cape_cin(p_prof, T_prof, Td_prof)
                sbcape_arr[j, i] = sbcape.to("J/kg").magnitude

            except Exception:
                pass

            pbar.update(1)

sbcape = xr.DataArray(
    sbcape_arr,
    coords={"latitude": lat, "longitude": lon},
    dims=("latitude", "longitude"),
    name="sbcape",
    attrs={"long_name": "Surface-based CAPE", "units": "J kg^-1"},
)
print(sbcape)

Calculating SBCAPE:   0%|          | 0/48081 [00:00<?, ?it/s]

/tmp/ipykernel_42409/3790612589.py:111: UserWarning: Interpolation point out of data bounds encountered
  sbcape, _ = mpcalc.surface_based_cape_cin(p_prof, T_prof, Td_prof)


<xarray.Dataset> Size: 17MB
Dimensions:         (valid_time: 1, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 8B 2026-03-11
  * latitude        (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    number          int64 8B 0
    pressure_level  float64 8B 500.0
    expver          <U4 16B '0005'
Data variables:
    z500            (valid_time, latitude, longitude) float32 4MB 4.954e+03 ....
    pwat            (valid_time, latitude, longitude) float32 4MB ...
    sbcape          (latitude, longitude) float64 8MB nan nan nan ... nan nan


# Step 4: remove unneeded lat/lon values, and combine to one dataset

In [28]:
wlon = 220
elon = 305
nlat = 55
slat = 20

# Subset first
z500_sub = z500.sel(
    longitude=slice(wlon, elon),
    latitude=slice(nlat, slat)
).squeeze("valid_time")

pwat_sub = pwat.sel(
    longitude=slice(wlon, elon),
    latitude=slice(nlat, slat)
).squeeze("valid_time")

## Step 5: Make a DataArray, then match the grid to the SOM's base dataset

In [29]:
combined = xr.Dataset(
    {
        "gh": z500_sub,
        "pwat": pwat_sub,
        "cape": sbcape,
    }
)

In [7]:
ds_gefs  = ds_gefs.sortby("time")

ds_era_coarse = ds_era.reindex(
    latitude=ds_gefs.latitude,
    longitude=ds_gefs.longitude,
    method="nearest"
)

ds_era_coarse.to_netcdf("/home/scratch/dwefer/era5/3-10-26_era5.nc")